# ✋ Touchless HCI: Real-Time Hand Tracking
**Project Type:** MediaPipe Hand Landmarker (Pretrained, No Training Required)

**Team:** Ahmed Khalid, Omar A. El Nasser, Bassel Adel, Karim Hossam, Mohamed Haitham, Omar Bekhiet

---

## 📖 Quick Setup & Execution Guide
Welcome to the inference notebook for the Touchless HCI Hand Tracker.

This version replaces the custom-trained YOLO hand detector with Google's **MediaPipe Hand Landmarker** task.
Because MediaPipe ships a high-quality model pretrained on a large hand dataset, the Roboflow dataset
download, the V1/V2/V3 training loops, and the ONNX export are no longer needed — we only need to
download the `.task` model file once and run inference.

MediaPipe also gives us **21 3D hand landmarks per hand** (not just a bounding box), which is a strictly
better signal for a touchless HCI gesture system than a YOLO bounding box was.

### ☁️ Option A: Google Colab
Just run the cells top to bottom. The webcam cell (last one) will **not** work in Colab — see Option B.

### 💻 Option B: Local (VSCode/PyCharm) — required for the live webcam demo
1. Create/activate a virtual environment.
2. Run the install cell below.
3. Run the model download cell.
4. Run the live webcam cell locally.

---
## 📦 1. Install Dependencies
*(MediaPipe ships its own inference runtime — no PyTorch/Ultralytics/Roboflow needed anymore)*

In [ ]:
!pip install mediapipe opencv-python numpy

## ⬇️ 2. Download the Pretrained Hand Landmarker Model
*(Downloads the official MediaPipe `hand_landmarker.task` bundle — a single pretrained model file,
replacing the entire Roboflow dataset + YOLO training pipeline)*

In [ ]:
import os
import urllib.request

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Downloading hand_landmarker.task ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Done.")
else:
    print("Model already downloaded.")

## 🎨 3. Landmark Drawing Helper
*(Draws the 21 hand landmarks + connections + handedness label on a frame, replacing YOLO's
`results[0].plot()` bounding-box renderer)*

In [ ]:
import cv2
import numpy as np
import mediapipe as mp

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green (BGR)

# Standard 21-point hand connections (index pairs), same topology MediaPipe uses.
# Defined manually because mp.solutions (legacy API) has been removed from
# recent mediapipe releases -- we no longer rely on it for drawing.
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),          # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),          # index finger
    (5, 9), (9, 10), (10, 11), (11, 12),     # middle finger
    (9, 13), (13, 14), (14, 15), (15, 16),   # ring finger
    (13, 17), (17, 18), (18, 19), (19, 20),  # pinky
    (0, 17),                                 # palm base
]


def draw_landmarks_on_image(bgr_image, detection_result):
    """Draws hand landmarks + handedness label onto a BGR (OpenCV-style) image."""
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(bgr_image)
    height, width, _ = annotated_image.shape

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        handedness = handedness_list[idx]

        points = [(int(lm.x * width), int(lm.y * height)) for lm in hand_landmarks]

        # Draw connections (skeleton)
        for start_idx, end_idx in HAND_CONNECTIONS:
            cv2.line(annotated_image, points[start_idx], points[end_idx], (255, 255, 255), 2)

        # Draw landmark points
        for (x, y) in points:
            cv2.circle(annotated_image, (x, y), 4, (0, 128, 255), -1)
            cv2.circle(annotated_image, (x, y), 4, (0, 0, 0), 1)

        # Handedness label above the hand
        x_coords = [p[0] for p in points]
        y_coords = [p[1] for p in points]
        text_x = min(x_coords)
        text_y = min(y_coords) - MARGIN

        cv2.putText(
            annotated_image,
            f"{handedness[0].category_name}",
            (text_x, text_y),
            cv2.FONT_HERSHEY_DUPLEX,
            FONT_SIZE,
            HANDEDNESS_TEXT_COLOR,
            FONT_THICKNESS,
            cv2.LINE_AA,
        )

    return annotated_image

## 📸 4. Inference: Test on a Static Image
*(Runs the Hand Landmarker in `IMAGE` mode against a single test image — the MediaPipe equivalent
of the old `model.predict(...)` cell)*

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# --- Build a detector for single-image inference ---
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
image_options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
image_detector = mp_vision.HandLandmarker.create_from_options(image_options)

# --- Download a sample test image (swap for any local path if you prefer) ---
TEST_IMAGE_URL = "https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg"
TEST_IMAGE_PATH = "test_image.jpg"
urllib.request.urlretrieve(TEST_IMAGE_URL, TEST_IMAGE_PATH)

mp_image = mp.Image.create_from_file(TEST_IMAGE_PATH)
detection_result = image_detector.detect(mp_image)

annotated_bgr = draw_landmarks_on_image(cv2.cvtColor(mp_image.numpy_view(), cv2.COLOR_RGB2BGR), detection_result)

print(f"Hands detected: {len(detection_result.hand_landmarks)}")
cv2.imwrite("annotated_test_image.png", annotated_bgr)
print("Saved annotated result to annotated_test_image.png")

## 🎥 5. Inference: Live Webcam Tracking
*(Initializes OpenCV to capture live video frames and draw the 21 hand landmarks in real-time.
Uses the `VIDEO` running mode, which lets MediaPipe use frame-to-frame tracking for lower latency —
the same job the old `best.onnx` YOLO model did, just with richer landmark output.
**Must be run locally in VSCode/PyCharm, not Colab**)*

In [ ]:
import time

# --- Build a detector for video-stream inference ---
video_options = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=mp_vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
video_detector = mp_vision.HandLandmarker.create_from_options(video_options)

cap = cv2.VideoCapture(0)
print("Starting webcam... Press 'q' to exit.")

start_time = time.time()

while cap.isOpened():

    success, frame = cap.read()
    if not success:
        print("Failed to grab frame.")
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_frame = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    timestamp_ms = int((time.time() - start_time) * 1000)

    result = video_detector.detect_for_video(mp_frame, timestamp_ms)

    annotated_frame = draw_landmarks_on_image(frame, result)

    cv2.imshow('Touchless HCI - Live Hand Tracking (MediaPipe)', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
video_detector.close()